# MultimodalVA — Start Here

Cause-of-death classification from verbal autopsy (VA) data, in a notebook.
No prior machine-learning experience assumed.

Everything here goes through **one function**, `mmva.run()`. The steps are:

1. Load data
2. Look at the data before modelling
3. Light preprocessing
4. A model that reads the **narrative** (text)
5. A model that reads the **symptom indicators** (tabular)
6. Compare the results
7. Save and re-use the trained model
8. Publish to the Hugging Face Hub

**To use your own data**, you change four things and nothing else:
the file you load, and the names of your label, narrative and indicator
columns. Each cell below marks where.

The companion notebook `02_combine_text_and_tabular_fusion.ipynb` uses the
narrative and the indicators together.

> Install once:
> `pip install git+https://github.com/y-chu/MultimodalVA.git`
> The default install covers everything in this notebook. Add
> `"multimodalva[all]"` only for LoRA fine-tuning and AutoMM feature fusion.

## 1. Load data

`mmva.data()` returns ready-to-use **synthetic** example datasets so you can run this notebook with no files of your own. Replace it with `pd.read_csv("your_data.csv")` when you have real data — everything below is identical.

In [ ]:
import multimodalva as mmva
import pandas as pd

# See what synthetic datasets ship with the package:
mmva.list_datasets()

In [ ]:
# A typical VA record: a free-text narrative + InterVA "i-code" symptom
# indicators (yes/no) + a cause-of-death label.
df = mmva.data("va_sample", n_per_class=40)

# --- with your own data instead, just do: ---
# df = pd.read_csv("your_data.csv")

print(df.shape)
df.head(3)

## 2. Quick EDA

Understand the label distribution, the text, and missingness before modelling.

In [ ]:
LABEL = "cause_of_death"
TEXT  = "narrative"

# Class balance
df[LABEL].value_counts()

In [ ]:
# Narrative length (words) — informs the text model's max_length
lengths = df[TEXT].fillna("").str.split().map(len)
print("word count — min/median/max:", lengths.min(), int(lengths.median()), lengths.max())

# The tabular feature columns are the InterVA i-codes (i019a, i077o, ...)
feature_cols = [c for c in df.columns if c.startswith("i") and c[1:4].isdigit()]
print(f"{len(feature_cols)} symptom-indicator features, e.g. {feature_cols[:6]}")

# Missingness at a glance
df[feature_cols].isna().mean().sort_values(ascending=False).head()

## 3. Preprocessing

Do any cleaning/filtering you like here with plain pandas. `run()` also drops rows with a missing label (or missing text, for text tasks) automatically, so this stays minimal. Example: keep adults only if your data has an age column.

In [ ]:
# Example filters you might apply (this synthetic df has no age column, so skip):
# df = df[df["age_grp3"] == "adult"].reset_index(drop=True)

# Confirm the columns we'll model on
print("label      :", LABEL)
print("text column:", TEXT)
print("n features :", len(feature_cols))
print("n rows     :", len(df))

## 4. Unimodal text model

Fine-tune a transformer on the narrative. `mmva.run()` handles split → tokenise → (HPO) → train → predict → save.

Everything lands under `output_dir/`: `final/` (the model), `predictions/`, and (when you pass `hyperparams=Optimize(...)`) `hpo/` + diagnostics.

In [ ]:
text_result = mmva.run(
    task="text",
    data=df,
    label_col=LABEL,
    text_col=TEXT,
    model="bluebert",              # any alias from `multimodalva list-models`, a HF id, or a local dir
    output_dir="runs/text_bluebert",
    # --- kept light so it runs on a laptop; scale up for real work ---
    # hyperparams=mmva.Optimize(n_trials=30),   # uncomment to search instead
    # n_trials=30, optimize_metric="f1_macro",
    hyperparams={"epochs": 3, "batch_size": 16, "learning_rate": 2e-5},
    use_lora=False,
)

## 5. Unimodal tabular model

Same call, `task="tabular"`, over the symptom indicators. `features` accepts an explicit list, a regex (`"re:^i\\d{3}"`), or `"auto"` (every column except label/text).

In [ ]:
tab_result = mmva.run(
    task="tabular",
    data=df,
    label_col=LABEL,
    features=feature_cols,         # or r"re:^i\d{3}[a-zA-Z]$", or "auto"
    model="lightgbm",              # catboost | xgboost | random_forest | mlp | svm | knn | ...
    output_dir="runs/tabular_lightgbm",
    # hyperparams=mmva.Optimize(n_trials=40),   # uncomment to search instead
    encode_categoricals="ordinal",
)

## 6. Inspect results

Every `run()` returns the same result dict. `predictions` is a `PredictionResult` with `.top1`, `.full` (probabilities), `.topk`, and `.id2label`.

In [ ]:
from multimodalva.utils.metrics import score_predictions

for name, res in [("text/bluebert", text_result), ("tabular/lightgbm", tab_result)]:
    top1 = res["predictions"].top1
    acc = score_predictions(top1, "accuracy")
    f1  = score_predictions(top1, "f1_macro")
    csmf = score_predictions(top1, "csmf_accuracy")
    print(f"{name:20s}  accuracy={acc:.3f}  f1_macro={f1:.3f}  csmf={csmf:.3f}")

# Peek at the per-record predictions
text_result["predictions"].top1.head()

In [ ]:
# Nicely rendered leaderboard + a top-k accuracy plot (optional visuals)
from multimodalva.results import performance_leaderboard, plot_topk_accuracy

board = pd.DataFrame({
    "true_label": text_result["predictions"].top1["true_label"],
    "text_bluebert": text_result["predictions"].top1["predicted_label"],
})
performance_leaderboard(df=board, true_col="true_label",
                        model_cols=["text_bluebert"], sort_by="f1_macro")

## 7. Save & re-use the trained model

`run()` already saved everything to `output_dir/`. The text model is in standard Hugging Face format (cause labels baked into `config.json`), so you can reload it directly and classify a new narrative — no retraining, no package needed for inference.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_dir = "runs/text_bluebert/final"
tok = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir).eval()

narrative = "adult male with cough and fever for nine days before death"
with torch.no_grad():
    probs = model(**tok(narrative, return_tensors="pt", truncation=True)).logits.softmax(-1)
print("predicted cause:", model.config.id2label[int(probs.argmax())])

To continue **fine-tuning** the saved model on more data later, just pass its directory as `model` (a fresh head is initialised if your cause set differs):

In [ ]:
# mmva.run(task="text", data=more_df, label_col=LABEL, text_col=TEXT,
#          model="runs/text_bluebert/final", output_dir="runs/text_finetuned",
#          hyperparams={"epochs": 3, "ignore_mismatched_sizes": True})

## 8. Publish to the Hugging Face Hub

Text-based models are saved in standard HF format, so they can be pushed to the Hub and reloaded anywhere (text and data-fusion models only). Authenticate first with `huggingface-cli login`, or pass `hub_token=...`.

You can publish in one shot from `run()`:

In [ ]:
# One-shot: train and publish together
# result = mmva.run(
#     task="text", data=df, label_col=LABEL, text_col=TEXT, model="bluebert",
#     output_dir="runs/text_for_hub",
#     push_to_hub=True,
#     hub_repo_id="your-username/va-text-bluebert",
#     hub_private=True,          # set False to publish publicly
# )
# print(result["hub_url"])

In [ ]:
# Or publish a model you trained earlier, from its saved directory:
# from multimodalva.utils.hub import push_to_hub
# url = push_to_hub(
#     "runs/text_bluebert/final",
#     repo_id="your-username/va-text-bluebert",
#     model_kind="text",
#     private=True,
# )
# print(url)

---

**Next:** open `02_combine_text_and_tabular_fusion.ipynb` to use the narrative
and the symptom indicators together — data fusion, feature fusion and decision
fusion (voting, stacking).

`03_evaluate_and_compare_results.ipynb` then covers what to do with the output:
comparing several models, confidence intervals, confusion matrices and the rest.

**To run everything here as a script instead of a notebook**, see
`04_python_script_template.py`; to run it without writing Python at all, see
`05_command_line_template.sh` or `06_config_file_template.sh`.